# Load one hole: processed TSG or raw spectra

Choose a hole already downloaded on your computer. Run cells from top to bottom with the **HyLogger Data Crunch** kernel. Change `SOURCE` to switch between the two examples.

- **processed**: TSG-derived interpretation logs from NVCL Data Services, downloaded as CSV and Parquet.
- **raw**: measured spectra from NCI THREDDS ZIP packages. The reader uses the extracted `.tsg` headers and `.bip` binary spectra; you do not need to open these manually.

This notebook loads and previews inputs for clustering. It does not download data or fit a model.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from hylogger.config import AppConfig
from hylogger.prepare import prepare_hole

HOLE_ID = "05GJD001"
SOURCE = "processed"  # "processed" or "raw"
REGION = "nir"        # Raw only: "nir" (NIR/SWIR), "tir", or "mir", if available
DATA_DIR = None        # Or r"C:\HyLoggerData" if downloads live outside the repository

config = AppConfig.load(data_dir=DATA_DIR)
HOLE_ID = HOLE_ID.upper()
print("Data folder:", config.data_dir)
print("Hole:", HOLE_ID, "| Source:", SOURCE)


## Prepare the downloaded hole

`prepare_hole` is the Python equivalent of `hylogger prepare HOLE_ID --source processed` (or `--source raw`). It locates the downloaded files and writes prepared tables, features and a QC report. Re-running replaces prepared outputs for this hole and source; original downloads are preserved.

If data is missing, download it in the terminal first:

```powershell
.\.venv\Scripts\hylogger.exe processed fetch 05GJD001 --profile tsg_swir
```

For raw data, first estimate the full ZIP download, then download/extract only if you want it:

```powershell
.\.venv\Scripts\hylogger.exe raw fetch 05GJD001 --components spectra --dry-run
.\.venv\Scripts\hylogger.exe raw fetch 05GJD001 --components spectra
```

Use the same data folder in the terminal and notebook. Setting an environment variable in a terminal does not update an already-running notebook kernel; set `DATA_DIR` above explicitly if needed.


In [ ]:
result = prepare_hole(config, HOLE_ID, source=SOURCE)
print("Prepared source:", result["source"])
print("QC report:", result["qc_path"])
for path in result["sample_paths"]:
    print("Sample table:", path)


## Load and preview the inputs

Processed: `samples` contains depth-sorted logs, and `hole_features` contains one summary row for the whole hole.

Raw: `spectra` contains one row per measurement and one column per wavelength channel. `wavelength` describes those channels. Spectral regions remain separate.


In [ ]:
if SOURCE == "processed":
    samples = pd.read_parquet(result["sample_paths"][0])
    hole_features = pd.read_parquet(result["feature_path"])
    display(samples.head())
    print("Available columns:", samples.columns.tolist())
    display(hole_features)
else:
    with np.load(result["spectra_path"], allow_pickle=False) as archive:
        print("Available arrays:", archive.files)
        key = f"{REGION}_spectra"
        if key not in archive.files:
            raise ValueError(f"No {REGION} spectra. Choose a region listed above.")
        spectra = archive[key].astype(float)
        wavelength = archive[f"{REGION}_wavelength"].copy()
    print("Spectra shape (measurements, channels):", spectra.shape)
    print("Wavelength shape:", wavelength.shape)
    display(pd.DataFrame(spectra[:5, :10]))


## Starting point for clustering

For processed TSG, the first team baseline clusters **whole holes by group-label proportions**. It does not cluster category names as numbers. Below we inspect the relevant log; the existing baseline notebook handles valid/invalid labels and constructs the model matrix across multiple holes.

For raw spectra, below we build a small exploratory numeric matrix `X` using at most 1,000 readings. These are measurement rows, not whole holes. This is an input example, not a completed quality-control pipeline: review scanner masks, spectral quality and normalization before fitting or interpreting clusters.


In [ ]:
if SOURCE == "processed":
    GROUP_LOG = "grp1_u_tsas"
    if GROUP_LOG not in samples:
        raise ValueError(f"Missing {GROUP_LOG}. Check the downloaded log profile.")
    display(samples[GROUP_LOG].value_counts(dropna=False).rename("reading_count"))
else:
    if spectra.ndim != 2:
        raise ValueError("Expected a two-dimensional spectral matrix.")
    candidate = spectra[:1000].copy()
    candidate[candidate <= -1e38] = np.nan  # TSG missing-value sentinels
    valid = np.isfinite(candidate).all(axis=1)
    spectrum_row = np.flatnonzero(valid)  # Original spectral row indices
    X = candidate[valid]
    print("Exploratory X shape:", X.shape)
    print("Excluded non-finite readings:", int((~valid).sum()))
    if len(X) == 0:
        raise ValueError("No complete readings in this preview; inspect missing channels.")


## Continue with the team baseline

For the processed example, prepare the provided five-hole selection in the terminal:

```powershell
.\.venv\Scripts\hylogger.exe prepare --selection tsg_pilot --source processed
```

Then open `../30_clustering/01_baseline_clustering.ipynb` and set `SELECTION = "tsg_pilot"`. Follow the README's first processed TSG baseline guide.

For raw experiments, `X` is your exploratory spectral input. Keep `spectrum_row` to identify its original spectral rows. Do **not** join these row indices directly to depth-sorted prepared sample tables: the current preparation code sorts sample tables but retains spectra in their original order. Verify raw-header alignment before attaching depths or interpretation labels to spectral clusters.
